# Week 3: Soft Regime Gating

In this notebook, we train a **Gating Network** to softly assign ocean regions to different "Experts".

## Goal
- Replace hard K-Means boundaries with soft, smooth probabilities ($\\pi_k$).
- Visualize dynamic regime shifts (fronts).
- Improve MSE/R² by allowing soft mixing of physical laws.

## Method
We minimize:
$$ \\mathcal{L} = \\text{MSE}(y, \\sum_k \\pi_k(x) \\cdot \\hat{y}_k(x)) + \\lambda \\cdot \\mathcal{L}_{smoothness}(\\pi) $$

Where $\\hat{y}_k(x)$ are the predictions from our experts (e.g., Linear Models or Symbolic Equations discovered in Week 2).

In [ ]:
import sys
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import xarray as xr
import matplotlib.pyplot as plt
from tqdm import tqdm

sys.path.append(os.path.abspath('..'))

from scripts.preprocess import TRAIN_OUTPUT_PATH, TEST_OUTPUT_PATH
from src.climate_discovery.models.gating import GatingNetwork
from src.climate_discovery.models.mixture import MixtureOfExperts
from src.climate_discovery.models.loss import RegimeConsistencyLoss
from src.climate_discovery.models.symbolic import KMeansSymbolicRegressor

## 1. Load Data & Prepare Experts

In [ ]:
# Load Data
ds_train = xr.open_dataset(TRAIN_OUTPUT_PATH)

# Convert to Tensor directly for PyTorch
def to_tensors(ds, target_var='fco2'):
    df = ds.to_dataframe().reset_index().dropna()
    
    # Features for EXPERTS (SST, SSS, etc.)
    x_expert_cols = ['sst', 'sss', 'log_chl'] 
    
    # Features for GATING (Space + Time)
    # We normalize Lat/Lon to [-1, 1] approximately for NN stability
    df['lat_norm'] = df['lat'] / 90.0
    df['lon_norm'] = df['lon'] / 180.0
    x_gate_cols = ['lat_norm', 'lon_norm', 'sin_month', 'cos_month', 'year_feature']
    
    X_expert = torch.tensor(df[x_expert_cols].values, dtype=torch.float32)
    X_gate = torch.tensor(df[x_gate_cols].values, dtype=torch.float32)
    y = torch.tensor(df[target_var].values, dtype=torch.float32)
    
    return X_expert, X_gate, y, df

X_expert, X_gate, y, df = to_tensors(ds_train)

# --- DEFINE EXPERTS ---
# For this demo, let's fit simple Linear Regressions per K-Means cluster to act as "Experts"
# (This simulates having found Symbolic Laws, but is faster/cleaner for the gating training step)
from sklearn.cluster import KMeans
from sklearn.linear_model import LinearRegression

N_CLUSTERS = 3

# 1. Cluster
kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=42)
labels = kmeans.fit_predict(X_expert)

# 2. Fit Linear Experts
expert_models = []
expert_preds_full = np.zeros((len(y), N_CLUSTERS)) # Pre-compute all expert predictions

for k in range(N_CLUSTERS):
    mask = (labels == k)
    model = LinearRegression()
    model.fit(X_expert[mask], y[mask])
    expert_models.append(model)
    
    # Predict everywhere (what would this expert say if it ruled the whole ocean?)
    expert_preds_full[:, k] = model.predict(X_expert)

expert_preds_tensor = torch.tensor(expert_preds_full, dtype=torch.float32)

print(f"Prepared {N_CLUSTERS} Experts (Linear Proxies).")

## 2. Train Gating Network

In [ ]:
# Initialize Models
gating_net = GatingNetwork(input_dim=X_gate.shape[1], num_regimes=N_CLUSTERS, hidden_dim=64)
moe = MixtureOfExperts(gating_net, []) # We use precomputed predictions
optimizer = optim.Adam(gating_net.parameters(), lr=0.001)
criterion = nn.MSELoss()

# Training Loop
BATCH_SIZE = 1024
EPOCHS = 10
dataset = torch.utils.data.TensorDataset(X_gate, expert_preds_tensor, y)
loader = torch.utils.data.DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

losses = []

print("Training Soft Gating...")
for epoch in range(EPOCHS):
    epoch_loss = 0
    for xg_batch, exp_batch, y_batch in loader:
        optimizer.zero_grad()
        
        # Forward
        y_hat, probs = moe.forward_with_precomputed(xg_batch, exp_batch)
        
        # Loss (MSE only for now, TV/Smoothness comes next)
        loss = criterion(y_hat, y_batch)
        
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
    
    avg_loss = epoch_loss / len(loader)
    losses.append(avg_loss)
    print(f"Epoch {epoch+1}/{EPOCHS}, Loss: {avg_loss:.4f}")

plt.plot(losses)
plt.title("Training Loss")
plt.xlabel("Epoch")
plt.show()

## 3. Visualize Regimes (Soft vs Hard)
Here we visualize the probabilities $\pi_k$ on a map to see if we get coherent structures.

In [ ]:
# Predict on whole dataset for visualization
with torch.no_grad():
    gating_net.eval()
    _, probs = gating_net(X_gate)
    regime_probs = probs.numpy()
    
# Assign Soft Regimes back to DataFrame
for k in range(N_CLUSTERS):
    df[f'prob_k{k}'] = regime_probs[:, k]

df['hard_regime'] = np.argmax(regime_probs, axis=1)

# Convert back to XArray for easy plotting
ds_vis = df.set_index(['time', 'lat', 'lon']).to_xarray()

# Plot
print("Visualizing Regime Probabilities (Epoch 10)...")
fig, ax = plt.subplots(1, 3, figsize=(18, 5))

for k in range(N_CLUSTERS):
    # Plot average probability map for Regime K
    ds_vis[f'prob_k{k}'].mean(dim='time').plot(ax=ax[k], cmap='viridis', vmin=0, vmax=1)
    ax[k].set_title(f"Regime {k} Probability")

plt.tight_layout()
plt.show()

# Visualize Hard Labels for comparison
plt.figure(figsize=(10, 5))
ds_vis['hard_regime'].isel(time=0).plot(cmap='tab10')
plt.title("Hard Regime Map (Derived from Soft Gating)")
plt.show()

## 4. Quantitative Comparison: Hard vs Soft
We compare the $R^2$ of the Mixture of Experts (Soft) vs the Hard K-Means assignment.

In [ ]:
from sklearn.metrics import r2_score

# 1. Soft Prediction
with torch.no_grad():
    gating_net.eval()
    # Forward pass: weights * experts
    y_soft, _ = moe.forward_with_precomputed(X_gate, expert_preds_tensor)
    y_soft = y_soft.numpy()

# 2. Hard Prediction (Best Single Expert per point)
# expert_preds_full is (N, K)
hard_labels = np.argmax(regime_probs, axis=1)
y_hard = expert_preds_full[np.arange(len(y)), hard_labels]

# 3. Metrics
r2_soft = r2_score(y.numpy(), y_soft)
r2_hard = r2_score(y.numpy(), y_hard)

print(f"Hard K-Means R2: {r2_hard:.4f}")
print(f"Soft Gating R2:  {r2_soft:.4f}")
print(f"Improvement:     {r2_soft - r2_hard:.4f}")

if r2_soft > r2_hard:
    print("✅ Soft Gating improved performance!")
else:
    print("⚠️ Soft Gating did not improve (might need more training or tuning).")